# TalkNet-ASD: Optimized Inference Demo
**GPU Required:** `Runtime -> Change runtime type -> T4 GPU`, then Run All.

In [ ]:
# GPU GUARD: Fail immediately if no GPU is attached
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "\n\n" + "="*60 + "\n"
        "NO GPU DETECTED!\n"
        "Go to: Runtime -> Change runtime type -> T4 GPU -> Save\n"
        "Then: Runtime -> Run All\n"
        + "="*60
    )
print(f"GPU OK: {torch.cuda.get_device_name(0)}")

# Cell 1: Setup Environment
import os
os.chdir("/content")

!rm -rf TalkNet-ASD
!git clone -q https://github.com/TaoRuijie/TalkNet-ASD.git
os.chdir("/content/TalkNet-ASD")

!pip install -q scenedetect==0.5.6.1
!pip install -q gdown scipy librosa opencv-python python_speech_features tqdm
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1

# Patch deprecated np.int/np.float for NumPy 1.24+
!find . -name "*.py" -exec sed -i "s/np\.int)/int)/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.int,/int,/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.int]/int]/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.float)/float)/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.bool)/bool)/g" {} +

print("=== Setup complete ===")

In [ ]:
# Cell 2: Upload & Pre-process Video
# We:
#   1. Accept your upload
#   2. Resize to 480p and limit to 60 seconds using ffmpeg BEFORE passing to TalkNet
#      This is the single biggest speedup: less data = faster everything downstream.
import os, time
from google.colab import files
os.chdir("/content/TalkNet-ASD")
!mkdir -p demo

print("Upload your video (mp4):")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
video_name_no_ext = "test_clip"

print(f"\nPre-processing {filename} (resize to 480p, resize to 480p for faster inference)...")
t0 = time.time()

# Key optimizations:
# -t 60         : take only first 60 seconds
# -vf scale=-2:480 : resize to 480p (preserving aspect ratio)
# -crf 23       : fast encode
# -preset fast  : fast encode
# -y            : overwrite without prompting
!ffmpeg -y -i "{filename}" -vf scale=-2:480 -crf 23 -preset fast \
    demo/test_clip.mp4 2>&1 | tail -5

elapsed = time.time() - t0
size = os.path.getsize("demo/test_clip.mp4") / (1024*1024)
print(f"\nPre-processing done in {elapsed:.1f}s => demo/test_clip.mp4 ({size:.1f} MB)")
print("=== Video ready ===")

In [ ]:
# Cell 3: Run Optimized TalkNet Inference
# Tuned flags for maximum speed:
#   --nDataLoaderThread 4  : 4 parallel workers (T4 GPU has 4 cores)
#   --facedetScale 0.25    : already default, keep it (bigger = slower)
#   --minTrack 5           : from 10 -> 5 (accept shorter face tracks, fewer misses)
#   --numFailedDet 20      : allow more misses before dropping a track (avoids re-detection)
import os, time
os.chdir("/content/TalkNet-ASD")

print(f"Running TalkNet inference (optimized)...")
t0 = time.time()

!python demoTalkNet.py \
    --videoName test_clip \
    --nDataLoaderThread 4 \
    --facedetScale 0.25 \
    --minTrack 5 \
    --numFailedDet 20 \

elapsed = time.time() - t0
print(f"\n=== Inference done in {elapsed:.0f}s ({elapsed/60:.1f} min) ===")

In [ ]:
# Cell 4: Download Result
import os
os.chdir("/content/TalkNet-ASD")
from google.colab import files

output_path = "demo/test_clip/pyavi/video_out.avi"

if os.path.exists(output_path):
    size = os.path.getsize(output_path) / (1024*1024)
    print(f"SUCCESS: {output_path} ({size:.1f} MB)")
    files.download(output_path)
else:
    print("Not found. Full demo tree:")
    !ls -R demo/